In [1]:
print("hello")

hello


## combine wether data files into one file

In [1]:
from pathlib import Path
import pandas as pd

# Change this to your folder.
INPUT_FOLDER = Path(r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\wether data\csv")
OUTPUT_FILE = Path(r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data") / "weather_combined_03 combine_wether_data.csv"

files = sorted(
    INPUT_FOLDER.glob("weather_part_*_0600_1900.csv")
)

if len(files) != 7:
    raise ValueError(f"Expected 7 part files, found {len(files)}")

frames = []
expected_columns = None

for path in files:
    df = pd.read_csv(
        path,
        dtype={"location_id": "string"},
        encoding="utf-8-sig",
    )

    if expected_columns is None:
        expected_columns = df.columns.tolist()
    elif set(df.columns) != set(expected_columns):
        raise ValueError(f"Columns do not match: {path.name}")

    frames.append(df[expected_columns])
    print(f"Added: {path.name} — {len(df):,} rows")

combined = pd.concat(frames, ignore_index=True)

# Parse UTC for reliable chronological sorting and duplicate checks.
combined["timestamp_utc"] = pd.to_datetime(
    combined["timestamp_utc"],
    utc=True,
    errors="raise",
)

duplicates = combined.duplicated(
    ["location_id", "timestamp_utc"],
    keep=False,
)

if duplicates.any():
    raise ValueError(
        f"Found {duplicates.sum():,} rows with duplicate "
        "location/timestamp keys. Check the input files."
    )

combined = combined.sort_values(
    ["location_id", "timestamp_utc"]
).reset_index(drop=True)

combined.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print(f"\nCombined rows: {len(combined):,}")
print(f"Locations: {combined['location_id'].nunique()}")
print(f"Saved: {OUTPUT_FILE}")

Added: weather_part_01_2023-06-01_2026-08-02_0600_1900.csv — 48,678 rows
Added: weather_part_02_2023-06-01_2026-08-02_0600_1900.csv — 48,678 rows
Added: weather_part_03_2023-06-01_2026-08-02_0600_1900.csv — 48,678 rows
Added: weather_part_04_2023-06-01_2026-08-02_0600_1900.csv — 48,678 rows
Added: weather_part_05_2023-06-01_2026-08-02_0600_1900.csv — 48,678 rows
Added: weather_part_06_2023-06-01_2026-08-02_0600_1900.csv — 48,678 rows
Added: weather_part_07_2023-06-01_2026-08-02_0600_1900.csv — 48,678 rows

Combined rows: 340,746
Locations: 21
Saved: C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\weather_combined_03 combine_wether_data.csv


## combine wether + solar device data

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# FILE PATHS
# ============================================================

SOLAR_FILE = Path(
    r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\final_solaronly_combined_01 preprocessing_solar.csv"
)

WEATHER_FILE = Path(
    r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\weather_combined_03 combine_wether_data.csv"
)

OUTPUT_FILE = Path(
    r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\solar_with_weather_03 combine_wether_data.csv"
)

CHUNK_SIZE = 100_000

# Units follow the earlier weather downloader:
# temperature: °C; wind: m/s; precipitation/rain: mm.
WEATHER_COLUMNS = {
    "temperature_2m": "weather_temperature_c",
    "relative_humidity_2m": "weather_relative_humidity_pct",
    "wind_speed_10m": "weather_wind_speed_ms",
    "cloud_cover": "weather_cloud_cover_pct",
    "cloud_cover_low": "weather_cloud_cover_low_pct",
    "cloud_cover_mid": "weather_cloud_cover_mid_pct",
    "cloud_cover_high": "weather_cloud_cover_high_pct",
    "shortwave_radiation_instant": "weather_ghi_instant_wm2",
    "direct_normal_irradiance_instant": "weather_dni_instant_wm2",
    "diffuse_radiation_instant": "weather_dhi_instant_wm2",
    "precipitation": "weather_preceding_hour_precipitation_mm",
    "rain": "weather_preceding_hour_rain_mm",
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def location_key(frame):
    """Create a location key from requested coordinates."""
    latitude = pd.to_numeric(frame["latitude"], errors="raise")
    longitude = pd.to_numeric(frame["longitude"], errors="raise")

    if (
        latitude.isna().any()
        or longitude.isna().any()
        or not latitude.between(-90, 90).all()
        or not longitude.between(-180, 180).all()
    ):
        raise ValueError("Missing or invalid latitude/longitude")

    return (
        latitude.map(lambda value: f"{value:.6f}")
        + "|"
        + longitude.map(lambda value: f"{value:.6f}")
    )


def parse_utc(series):
    """Parse the timezone-aware timestamps used in these CSVs."""
    parsed = pd.to_datetime(
        series,
        utc=True,
        errors="raise",
        format="mixed",
    )

    if parsed.isna().any():
        raise ValueError("Missing timestamps found")

    return parsed


# ============================================================
# CHECK INPUTS
# ============================================================

for path in (SOLAR_FILE, WEATHER_FILE):
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

if OUTPUT_FILE.resolve() in {
    SOLAR_FILE.resolve(),
    WEATHER_FILE.resolve(),
}:
    raise ValueError("Output must differ from both input files")

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

solar_header = pd.read_csv(
    SOLAR_FILE,
    nrows=0,
    encoding="utf-8-sig",
).columns

required_solar_columns = {
    "latitude",
    "longitude",
    "timestamp_5min",
}

missing = required_solar_columns - set(solar_header)

if missing:
    raise ValueError(f"Missing solar columns: {sorted(missing)}")

new_columns = {
    "solar_aligned_timestamp_utc",
    "weather_timestamp_utc",
    "weather_age_minutes",
    "weather_matched",
    "weather_features_complete",
    *WEATHER_COLUMNS.values(),
}

collisions = new_columns.intersection(solar_header)

if collisions:
    raise ValueError(
        "Solar input already contains output columns: "
        f"{sorted(collisions)}"
    )


# ============================================================
# PREPARE WEATHER DATA
# ============================================================

weather = pd.read_csv(
    WEATHER_FILE,
    usecols=[
        "timestamp_utc",
        "latitude",
        "longitude",
        *WEATHER_COLUMNS.keys(),
    ],
    encoding="utf-8-sig",
)

if weather.empty:
    raise ValueError("Weather file contains no data rows")

weather["_location_key"] = location_key(weather)

weather["weather_timestamp_utc"] = parse_utc(
    weather.pop("timestamp_utc")
)

weather = weather.rename(columns=WEATHER_COLUMNS)

for column in WEATHER_COLUMNS.values():
    weather[column] = pd.to_numeric(
        weather[column],
        errors="raise",
    )

weather_keys = [
    "_location_key",
    "weather_timestamp_utc",
]

if weather.duplicated(weather_keys).any():
    raise ValueError(
        "Duplicate weather location/timestamp pairs found. "
        "Resolve these before joining."
    )

weather_locations = set(weather["_location_key"])

weather = weather[
    [
        "_location_key",
        "weather_timestamp_utc",
        *WEATHER_COLUMNS.values(),
    ]
].sort_values("weather_timestamp_utc")

print(f"Weather rows: {len(weather):,}")
print(f"Weather locations: {len(weather_locations)}")


# ============================================================
# JOIN SOLAR DATA IN CHUNKS
# ============================================================

temporary_file = OUTPUT_FILE.with_suffix(".csv.tmp")

total_rows = 0
matched_rows = 0
complete_rows = 0
first_chunk = True

# Preserve original CSV values, particularly long identifiers.
solar_dtypes = {column: "string" for column in solar_header}

try:
    with pd.read_csv(
        SOLAR_FILE,
        dtype=solar_dtypes,
        encoding="utf-8-sig",
        chunksize=CHUNK_SIZE,
    ) as reader:

        for chunk in reader:
            chunk["_location_key"] = location_key(chunk)

            unknown_locations = (
                set(chunk["_location_key"])
                - weather_locations
            )

            if unknown_locations:
                raise ValueError(
                    "Solar coordinates absent from weather data: "
                    f"{sorted(unknown_locations)}"
                )

            chunk["solar_aligned_timestamp_utc"] = parse_utc(
                chunk["timestamp_5min"]
            )

            # Restore original row order after the time-based join.
            chunk["_original_order"] = np.arange(len(chunk))

            chunk = chunk.sort_values(
                "solar_aligned_timestamp_utc"
            )

            combined = pd.merge_asof(
                chunk,
                weather,
                left_on="solar_aligned_timestamp_utc",
                right_on="weather_timestamp_utc",
                by="_location_key",
                direction="backward",
                allow_exact_matches=True,
                tolerance=pd.Timedelta("59min 59s"),
            )

            combined["weather_age_minutes"] = (
                combined["solar_aligned_timestamp_utc"]
                - combined["weather_timestamp_utc"]
            ).dt.total_seconds() / 60

            combined["weather_matched"] = (
                combined["weather_timestamp_utc"].notna()
            )

            combined["weather_features_complete"] = (
                combined["weather_matched"]
                & combined[list(WEATHER_COLUMNS.values())]
                .notna()
                .all(axis=1)
            )

            if len(combined) != len(chunk):
                raise ValueError(
                    "Join unexpectedly changed the solar row count"
                )

            matched = combined["weather_matched"]

            if not combined.loc[
                matched, "weather_age_minutes"
            ].between(0, 60, inclusive="left").all():
                raise ValueError("Invalid weather time alignment")

            combined = (
                combined.sort_values("_original_order")
                .drop(columns=[
                    "_location_key",
                    "_original_order",
                ])
            )

            combined.to_csv(
                temporary_file,
                index=False,
                mode="w" if first_chunk else "a",
                header=first_chunk,
                encoding="utf-8",
            )

            first_chunk = False

            total_rows += len(combined)
            matched_rows += int(
                combined["weather_matched"].sum()
            )
            complete_rows += int(
                combined["weather_features_complete"].sum()
            )

            print(
                f"Processed {total_rows:,} rows | "
                f"weather matched: {matched_rows:,}"
            )

    if first_chunk:
        raise ValueError("Solar file contains no data rows")

    temporary_file.replace(OUTPUT_FILE)

finally:
    if temporary_file.exists():
        temporary_file.unlink()


# ============================================================
# SUMMARY
# ============================================================

print("\nFinished")
print(f"Solar rows preserved: {total_rows:,}")
print(f"Weather matched: {matched_rows:,}")
print(f"Weather unmatched: {total_rows - matched_rows:,}")
print(f"Complete weather features: {complete_rows:,}")
print(f"Saved to: {OUTPUT_FILE}")

Weather rows: 340,746
Weather locations: 21
Processed 100,000 rows | weather matched: 100,000
Processed 200,000 rows | weather matched: 200,000
Processed 300,000 rows | weather matched: 300,000
Processed 400,000 rows | weather matched: 400,000
Processed 500,000 rows | weather matched: 500,000
Processed 600,000 rows | weather matched: 600,000
Processed 700,000 rows | weather matched: 700,000
Processed 800,000 rows | weather matched: 800,000
Processed 900,000 rows | weather matched: 900,000
Processed 1,000,000 rows | weather matched: 1,000,000
Processed 1,100,000 rows | weather matched: 1,100,000
Processed 1,200,000 rows | weather matched: 1,200,000
Processed 1,300,000 rows | weather matched: 1,300,000
Processed 1,400,000 rows | weather matched: 1,400,000
Processed 1,500,000 rows | weather matched: 1,500,000
Processed 1,600,000 rows | weather matched: 1,600,000
Processed 1,700,000 rows | weather matched: 1,700,000
Processed 1,800,000 rows | weather matched: 1,800,000
Processed 1,900,000 

## combine sun position with pvlib

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import pvlib

INPUT_FILE = Path(
    r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\solar_with_weather_03 combine_wether_data.csv"
)

OUTPUT_FILE = Path(
    r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\solar_with_weather_sun_03 combine_wether_data.csv"
)

CHUNK_SIZE = 100_000

SUN_COLUMNS = [
    "sun_elevation_deg",
    "sun_zenith_deg",
    "sun_azimuth_deg",
    "sun_above_horizon",
    "sun_azimuth_sin",
    "sun_azimuth_cos",
]

if not INPUT_FILE.is_file():
    raise FileNotFoundError(INPUT_FILE)

if INPUT_FILE.resolve() == OUTPUT_FILE.resolve():
    raise ValueError("Output must differ from input")

header = pd.read_csv(
    INPUT_FILE,
    nrows=0,
    encoding="utf-8-sig",
).columns.tolist()

required = {"latitude", "longitude", "timestamp_5min"}

missing = required - set(header)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

collisions = set(SUN_COLUMNS).intersection(header)
if collisions:
    raise ValueError(
        f"Sun columns already exist: {sorted(collisions)}"
    )

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
temporary_file = OUTPUT_FILE.with_suffix(".csv.tmp")

total_rows = 0
first_chunk = True

try:
    with pd.read_csv(
        INPUT_FILE,
        dtype={column: "string" for column in header},
        encoding="utf-8-sig",
        chunksize=CHUNK_SIZE,
    ) as reader:

        for chunk in reader:
            # Keep original columns unchanged.
            chunk["_latitude"] = pd.to_numeric(
                chunk["latitude"], errors="raise"
            ).astype(float)

            chunk["_longitude"] = pd.to_numeric(
                chunk["longitude"], errors="raise"
            ).astype(float)

            chunk["_sun_time"] = pd.to_datetime(
                chunk["timestamp_5min"],
                utc=True,
                errors="raise",
                format="mixed",
            )

            if (
                chunk["_latitude"].isna().any()
                or chunk["_longitude"].isna().any()
                or chunk["_sun_time"].isna().any()
                or not chunk["_latitude"].between(-90, 90).all()
                or not chunk["_longitude"].between(-180, 180).all()
            ):
                raise ValueError(
                    "Missing or invalid coordinates/timestamps"
                )

            chunk["_row_order"] = np.arange(len(chunk))

            keys = ["_latitude", "_longitude", "_sun_time"]
            unique = chunk[keys].drop_duplicates()

            position_frames = []

            for (latitude, longitude), group in unique.groupby(
                ["_latitude", "_longitude"],
                sort=False,
            ):
                times = pd.DatetimeIndex(group["_sun_time"])

                position = pvlib.solarposition.get_solarposition(
                    time=times,
                    latitude=latitude,
                    longitude=longitude,
                    altitude=0,  # Approximate: site elevation unavailable.
                    method="nrel_numpy",
                )

                result = group.copy()

                # Geometric angles: these do not include
                # atmospheric-refraction corrections.
                result["sun_elevation_deg"] = (
                    position["elevation"].to_numpy()
                )
                result["sun_zenith_deg"] = (
                    position["zenith"].to_numpy()
                )
                result["sun_azimuth_deg"] = (
                    position["azimuth"].to_numpy()
                )

                result["sun_above_horizon"] = (
                    result["sun_elevation_deg"] > 0
                )

                azimuth_radians = np.deg2rad(
                    result["sun_azimuth_deg"]
                )

                result["sun_azimuth_sin"] = np.sin(
                    azimuth_radians
                )
                result["sun_azimuth_cos"] = np.cos(
                    azimuth_radians
                )

                position_frames.append(result)

            sun_lookup = pd.concat(
                position_frames,
                ignore_index=True,
            )

            combined = chunk.merge(
                sun_lookup,
                on=keys,
                how="left",
                validate="many_to_one",
            )

            if len(combined) != len(chunk):
                raise ValueError("Unexpected change in row count")

            if combined[SUN_COLUMNS].isna().any().any():
                raise ValueError("Missing calculated sun positions")

            combined = (
                combined.sort_values("_row_order")
                .drop(columns=[
                    "_latitude",
                    "_longitude",
                    "_sun_time",
                    "_row_order",
                ])
            )

            combined.to_csv(
                temporary_file,
                index=False,
                mode="w" if first_chunk else "a",
                header=first_chunk,
                encoding="utf-8",
            )

            first_chunk = False
            total_rows += len(combined)

            print(f"Processed {total_rows:,} rows")

    if first_chunk:
        raise ValueError("Input file contains no data rows")

    temporary_file.replace(OUTPUT_FILE)

finally:
    if temporary_file.exists():
        temporary_file.unlink()

print(f"\nFinished: {total_rows:,} rows")
print(f"Saved to: {OUTPUT_FILE}")


Processed 100,000 rows
Processed 200,000 rows
Processed 300,000 rows
Processed 400,000 rows
Processed 500,000 rows
Processed 600,000 rows
Processed 700,000 rows
Processed 800,000 rows
Processed 900,000 rows
Processed 1,000,000 rows
Processed 1,100,000 rows
Processed 1,200,000 rows
Processed 1,300,000 rows
Processed 1,400,000 rows
Processed 1,500,000 rows
Processed 1,600,000 rows
Processed 1,700,000 rows
Processed 1,800,000 rows
Processed 1,900,000 rows
Processed 2,000,000 rows
Processed 2,100,000 rows
Processed 2,200,000 rows
Processed 2,300,000 rows
Processed 2,400,000 rows
Processed 2,500,000 rows
Processed 2,600,000 rows
Processed 2,700,000 rows
Processed 2,769,082 rows

Finished: 2,769,082 rows
Saved to: C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\solar_with_weather_sun_03 combine_wether_data.csv


## normalice power 

In [4]:
infolist=[
{"file": "CRN 10049 Meepe,Habaraduwa_1.csv", "town": "Habaraduwa", "district": "Galle", "pv_capacity_kwp": 6.48, "latitude": 5.9977, "longitude": 80.2958},
{"file": "CRN 11630 Molligoda,Wadduwa (Convert)_1.csv", "town": "Wadduwa", "district": "Kalutara", "pv_capacity_kwp": 5, "latitude": 6.6640, "longitude": 79.9305}, 
{"file": "CRN 11183 Ambalangoda_1.csv", "town": "Ambalangoda", "district": "Galle", "pv_capacity_kwp": 15, "latitude": 6.2353, "longitude": 80.0550},
{"file": "CRN 12157.csv", "town": "Ambalangoda", "district": "Galle", "pv_capacity_kwp": 5.2, "latitude": 6.2353, "longitude": 80.0550},
{"file": "CRN 18434 Polwatta Road, Ambalangoda_1.csv", "town": "Ambalangoda", "district": "Galle", "pv_capacity_kwp": 5, "latitude": 6.2353, "longitude": 80.0550},
{"file": "CRN 52674 Porabha Ambalangoda_1.csv", "town": "Ambalangoda", "district": "Galle", "pv_capacity_kwp": 6, "latitude": 6.2353, "longitude": 80.0550},
{"file": "CRN 52674 Porabha Ambalangoda_2.csv", "town": "Ambalangoda", "district": "Galle", "pv_capacity_kwp": 6, "latitude": 6.2353, "longitude": 80.0550},
{"file": "CRN 52919 Wathugedara Ambalangoda_1.csv", "town": "Ambalangoda", "district": "Galle", "pv_capacity_kwp": 8, "latitude": 6.2353, "longitude": 80.0550},
{"file": "CRN 38794 Ambalangoda.csv", "town": "Ambalangoda", "district": "Galle", "pv_capacity_kwp": 3.1, "latitude": 6.2353, "longitude": 80.0550},


{"file": "CRN 12200 ( Hybrid )Milidduwa, Galle_1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 5, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN 14857 Karapitiya Galle_1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 6, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN 31575 Hirimbura Road, Galle_1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 3.2, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN 33061_1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 8, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN 34350 Walahanduwa,Galle_1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 3, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN 37399 Karaputiya.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 3.9, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN 39828 Anagarika dharmapala mawatha Galle_1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 6, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN 40400 Labuduwa, Gall_1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 5, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN 49244 Bope Cross Rd,Galle._1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 4.95, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN 53616 walahanduwa Galle_1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 10, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN 53652 halpotha Galle_1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 22.32, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN12243.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 5.1, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN14037 (Hybrid).csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 6, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN25287.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 1.95, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN35216_1.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 6.12, "latitude": 6.0329, "longitude": 80.2168},
{"file": "CRN35737 Mullegewaththa Kalahe Wanchawala.csv", "town": "Galle", "district": "Galle", "pv_capacity_kwp": 3.25, "latitude": 6.0329, "longitude": 80.2168},


{"file": "CRN 18649 Thottawatta Road, Ahungalla_1.csv", "town": "Ahungalla", "district": "Galle", "pv_capacity_kwp": 7.25, "latitude": 6.3117, "longitude": 80.0345},

{"file": "CRN 26198 ( Hybrid ) Weheragoda,Ahangama.csv", "town": "Ahangama", "district": "Galle", "pv_capacity_kwp": 5.5, "latitude": 5.9726, "longitude": 80.3653},
{"file": "CRN 52354_1.csv", "town": "Ahangama", "district": "Galle", "pv_capacity_kwp": 17, "latitude": 5.9726, "longitude": 80.3653},

{"file": "CRN 28970 Handigoda Hikkaduwa_1.csv", "town": "Hikkaduwa", "district": "Galle", "pv_capacity_kwp": 5, "latitude": 6.1396, "longitude": 80.1030},
{"file": "CRN 8644 Udumulla,Hikkaduwa._1.csv", "town": "Hikkaduwa", "district": "Galle", "pv_capacity_kwp": 5, "latitude": 6.1396, "longitude": 80.1030},
{"file": "CRN 962 Panvila, Pathana, Hikkaduwa_1.csv", "town": "Hikkaduwa", "district": "Galle", "pv_capacity_kwp": 5, "latitude": 6.1396, "longitude": 80.1030},

{"file": "CRN 32558 Pitiwella,Boossa_1.csv", "town": "Pitiwella", "district": "Galle", "pv_capacity_kwp": 5, "latitude": 6.0712, "longitude": 80.1691},

{"file": "CRN 39939 Thawaluwila, Ambalantota (Convert)_1.csv", "town": "Ambalantota", "district": "Hambantota", "pv_capacity_kwp": 5, "latitude": 6.1225, "longitude": 81.0239},
{"file": "CRN 40774 Payagala Kaluthara_1.csv", "town": "Payagala", "district": "Kalutara", "pv_capacity_kwp": 20, "latitude": 6.5683, "longitude": 79.9825},
{"file": "CRN 41208 mathugama pradeshiya saba_1.csv", "town": "Matugama", "district": "Kalutara", "pv_capacity_kwp": 6, "latitude": 6.5222, "longitude":  80.1147},
{"file": "CRN 52890 Molligoda Wadduwa_1.csv", "town": "Wadduwa", "district": "Kalutara", "pv_capacity_kwp": 5, "latitude": 6.6640, "longitude": 79.9305},

{"file": "CRN 53084, K K M R I W D K Purijjala, Sella Katharagama_1.csv", "town": "Sella Kataragama", "district": "Monaragala", "pv_capacity_kwp": 27.5, "latitude": 6.4442, "longitude": 81.3090},
{"file": "CRN 53204 Mailagama Katharagama_1.csv", "town": "Sella Kataragama", "district": "Monaragala", "pv_capacity_kwp": 16.52, "latitude": 6.4442, "longitude": 81.3090},

{"file": "CRN 53200 Meegahawela road Beliattha_1.csv", "town": "Beliatta", "district": "Hambantota", "pv_capacity_kwp": 15.4, "latitude": 6.0394, "longitude": 80.7445},
{"file": "CRN 53444 Thissapura Uhana ampara_1.csv", "town": "Uhana", "district": "Ampara", "pv_capacity_kwp": 20, "latitude": 7.3323, "longitude": 81.6781},
{"file": "CRN 53689 Kurudu wattha Akmeemana_1.csv", "town": "Akmeemana", "district": "Galle", "pv_capacity_kwp": 5.58, "latitude": 6.0664, "longitude": 80.2589},
{"file": "CRN 53751 Mahaladuwa Balapitiya_1.csv", "town": "Balapitiya", "district": "Galle", "pv_capacity_kwp": 14.88, "latitude": 6.2731, "longitude": 80.0392},
{"file": "CRN 53974 (Weligama).csv", "town": "Weligama", "district": "Matara", "pv_capacity_kwp": 5, "latitude": 5.9722, "longitude": 80.4287},
{"file": "CRN.24542.csv", "town": "Kalametiya", "district": "Hambantota", "pv_capacity_kwp": 6.3, "latitude": 6.0839, "longitude": 80.9404},
{"file": "CRN13498 T Bay_1.csv", "town": "Tangalle", "district": "Hambantota", "pv_capacity_kwp": 17.9, "latitude": 6.0243, "longitude": 80.7942},

{"file": "CRN31412 South Kalutara.csv", "town": "Kalutara", "district": "Kalutara", "pv_capacity_kwp": 4.96, "latitude": 6.5854, "longitude": 79.9607},
{"file": "CRN33200- Dr Niroshan.csv", "town": "Kalutara", "district": "Kalutara", "pv_capacity_kwp": 5, "latitude": 6.5854, "longitude": 79.9607},
{"file": "CRN38340 Nagoda Kaluthra South.csv", "town": "Kalutara", "district": "Kalutara", "pv_capacity_kwp": 5, "latitude": 6.5854, "longitude": 79.9607},

{"file": "CRN35857.csv", "town": "Weniyarawela", "district": "Galle", "pv_capacity_kwp": 4.4, "latitude": 6.1865, "longitude": 80.1652},
]

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_FOLDER = Path(
    r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data"
)


INPUT_FILE = BASE_FOLDER / "solar_with_weather_sun_03 combine_wether_data.csv"
OUTPUT_FILE = BASE_FOLDER / "solar_with_weather_sun_normalized_03 combine_wether_data.csv"

CHUNK_SIZE = 100_000

# Assumes pac contains watts. Confirm this from your source data.
PAC_TO_WATTS = 1.0

# ------------------------------------------------------------
# Prepare capacity lookup from your infolist.
# Match by source filename, not town or device-number order.
# ------------------------------------------------------------

metadata = pd.DataFrame(infolist)

required_metadata = {"file", "pv_capacity_kwp"}

if not required_metadata.issubset(metadata.columns):
    raise ValueError("infolist must contain file and pv_capacity_kwp")

metadata = metadata[["file", "pv_capacity_kwp"]].copy()
metadata["file"] = metadata["file"].astype("string").str.strip()

metadata["pv_capacity_kwp"] = pd.to_numeric(
    metadata["pv_capacity_kwp"],
    errors="raise",
)

if (
    metadata["file"].isna().any()
    or metadata["file"].eq("").any()
    or not np.isfinite(metadata["pv_capacity_kwp"]).all()
    or metadata["pv_capacity_kwp"].le(0).any()
):
    raise ValueError("Missing filename or invalid PV capacity in infolist")

conflicts = metadata.groupby("file")["pv_capacity_kwp"].nunique()

if conflicts.gt(1).any():
    raise ValueError(
        "Conflicting capacities for files: "
        f"{conflicts[conflicts.gt(1)].index.tolist()}"
    )

capacity_lookup = (
    metadata.drop_duplicates("file")
    .set_index("file")["pv_capacity_kwp"]
)

# ------------------------------------------------------------
# Check input.
# ------------------------------------------------------------

if not INPUT_FILE.is_file():
    raise FileNotFoundError(INPUT_FILE)

if INPUT_FILE.resolve() == OUTPUT_FILE.resolve():
    raise ValueError("Output must differ from input")

header = pd.read_csv(
    INPUT_FILE,
    nrows=0,
    encoding="utf-8-sig",
).columns.tolist()

if not {"file", "pac"}.issubset(header):
    raise ValueError("Input must contain file and pac")

added_columns = {
    "pv_capacity_kwp",
    "power_w_per_kwp",
}

if added_columns.intersection(header):
    raise ValueError(
        "Capacity or normalized-power columns already exist. "
        "Use the input before these columns were added."
    )

# ------------------------------------------------------------
# Add capacity and normalized power in chunks.
# ------------------------------------------------------------

temporary_file = OUTPUT_FILE.with_suffix(".csv.tmp")
total_rows = 0
first_chunk = True

try:
    with pd.read_csv(
        INPUT_FILE,
        dtype={column: "string" for column in header},
        encoding="utf-8-sig",
        chunksize=CHUNK_SIZE,
    ) as reader:

        for chunk in reader:
            source_files = chunk["file"].str.strip()
            capacity = source_files.map(capacity_lookup)

            if capacity.isna().any():
                unmatched = source_files.loc[
                    capacity.isna()
                ].drop_duplicates().tolist()

                raise ValueError(
                    "No capacity found for these filenames. "
                    "Check them against infolist: "
                    f"{unmatched}"
                )

            power = pd.to_numeric(
                chunk["pac"],
                errors="raise",
            ).astype(float)

            if not np.isfinite(power).all():
                raise ValueError("pac contains missing or infinite values")

            if power.lt(0).any():
                raise ValueError("Negative pac values require review")

            chunk["pv_capacity_kwp"] = capacity

            chunk["power_w_per_kwp"] = (
                power * PAC_TO_WATTS / capacity
            )

            chunk.to_csv(
                temporary_file,
                index=False,
                mode="w" if first_chunk else "a",
                header=first_chunk,
                encoding="utf-8",
            )

            first_chunk = False
            total_rows += len(chunk)

            print(f"Processed {total_rows:,} rows")

    if first_chunk:
        raise ValueError("Input contains no data rows")

    temporary_file.replace(OUTPUT_FILE)

finally:
    if temporary_file.exists():
        temporary_file.unlink()

print(f"\nFinished: {total_rows:,} rows")
print(f"Saved: {OUTPUT_FILE}")

Processed 100,000 rows
Processed 200,000 rows
Processed 300,000 rows
Processed 400,000 rows
Processed 500,000 rows
Processed 600,000 rows
Processed 700,000 rows
Processed 800,000 rows
Processed 900,000 rows
Processed 1,000,000 rows
Processed 1,100,000 rows
Processed 1,200,000 rows
Processed 1,300,000 rows
Processed 1,400,000 rows
Processed 1,500,000 rows
Processed 1,600,000 rows
Processed 1,700,000 rows
Processed 1,800,000 rows
Processed 1,900,000 rows
Processed 2,000,000 rows
Processed 2,100,000 rows
Processed 2,200,000 rows
Processed 2,300,000 rows
Processed 2,400,000 rows
Processed 2,500,000 rows
Processed 2,600,000 rows
Processed 2,700,000 rows
Processed 2,769,082 rows

Finished: 2,769,082 rows
Saved: C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\solar_with_weather_sun_normalized_03 combine_wether_data.csv


## 15 minute intervels

In [7]:
from pathlib import Path
import pandas as pd

BASE_FOLDER = Path(
    r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data"
)

INPUT_FILE = BASE_FOLDER / "solar_with_weather_sun_normalized_03 combine_wether_data.csv"
OUTPUT_FILE = BASE_FOLDER / "solar_15min_with_weather_sun_normalized_03 combine_wether_data.csv"

GENERATION_COLUMNS = [
    "pac",
    "psumCal",
    "power_w_per_kwp",
]

# ------------------------------------------------------------
# Load data.
# ------------------------------------------------------------

df = pd.read_csv(
    INPUT_FILE,
    dtype={
        "inverterSn": "string",
        "dataTimestamp": "string",
    },
    encoding="utf-8-sig",
)

original_columns = df.columns.tolist()

required = {
    "inverterSn",
    "timestamp_5min",
    *GENERATION_COLUMNS,
}

missing = required - set(df.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

if df.empty:
    raise ValueError("Input contains no data rows")

if df["inverterSn"].isna().any():
    raise ValueError("Missing inverter serial numbers")

# Temporary working table; no helper columns enter the output.
work = pd.DataFrame(index=df.index)

work["inverterSn"] = df["inverterSn"]

work["time"] = (
    pd.to_datetime(
        df["timestamp_5min"],
        utc=True,
        errors="raise",
        format="mixed",
    )
    .dt.tz_convert("Asia/Colombo")
)

if work["time"].isna().any():
    raise ValueError("Missing timestamps")

if not work["time"].eq(work["time"].dt.floor("5min")).all():
    raise ValueError("Some timestamps are not on the five-minute grid")

if work.duplicated(["inverterSn", "time"]).any():
    raise ValueError(
        "Duplicate inverter/timestamp pairs found. "
        "Resolve these before averaging."
    )

work["slot"] = work["time"].dt.floor("15min")

for column in GENERATION_COLUMNS:
    work[column] = pd.to_numeric(df[column], errors="raise")

# ------------------------------------------------------------
# Average generation within each inverter's 15-minute slot.
# ------------------------------------------------------------

means = work.groupby(
    ["inverterSn", "slot"]
)[GENERATION_COLUMNS].transform("mean")

# Retain only exact cutoff rows: 08:00, 08:15, 08:30, etc.
cutoff_rows = work["time"].eq(work["slot"])

result = df.loc[cutoff_rows].copy()

# Replace only generation columns with their slot averages.
for column in GENERATION_COLUMNS:
    result[column] = means.loc[cutoff_rows, column]

# Preserve original column names, order, and cutoff-row values.
result = result[original_columns]

result.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print(f"Input rows: {len(df):,}")
print(f"15-minute rows: {len(result):,}")
print(f"Columns preserved: {len(result.columns)}")
print(f"Saved: {OUTPUT_FILE}")

Input rows: 2,769,082
15-minute rows: 924,275
Columns preserved: 38
Saved: C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\solar_15min_with_weather_sun_normalized_03 combine_wether_data.csv


## ading time

time_of_day_sin and time_of_day_cos represent the time within a day. Together, they tell the model that 23:55 and 00:00 are close together, even though their numerical hour values look far apart.


day_of_year_sin and day_of_year_cos represent the position within a year. They tell the model that December 31 and January 1 are neighboring days, helping it learn seasonal patterns.

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd

BASE_FOLDER = Path(
    r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data"
)

FILES = [
    (
        BASE_FOLDER / "solar_with_weather_sun_normalized_03 combine_wether_data.csv",
        BASE_FOLDER / "solar_5min_with_time_features_03 combine_wether_data.csv",
    ),
    (
        BASE_FOLDER / "solar_15min_with_weather_sun_normalized_03 combine_wether_data.csv",
        BASE_FOLDER / "solar_15min_with_time_features_03 combine_wether_data.csv",
    ),
]

CHUNK_SIZE = 100_000

for input_file, output_file in FILES:
    if not input_file.is_file():
        raise FileNotFoundError(input_file)

    temporary_file = output_file.with_suffix(".csv.tmp")
    first_chunk = True
    total_rows = 0

    try:
        with pd.read_csv(
            input_file,
            dtype="string",
            encoding="utf-8-sig",
            chunksize=CHUNK_SIZE,
        ) as reader:

            for chunk in reader:
                local_time = pd.to_datetime(
                    chunk["timestamp_5min"],
                    utc=True,
                    format="mixed",
                    errors="raise",
                ).dt.tz_convert("Asia/Colombo")

                if local_time.isna().any():
                    raise ValueError(
                        f"Missing timestamps in {input_file.name}"
                    )

                minutes_of_day = (
                    local_time.dt.hour * 60
                    + local_time.dt.minute
                )

                daily_angle = (
                    2 * np.pi * minutes_of_day / 1440
                )

                days_in_year = np.where(
                    local_time.dt.is_leap_year, 366, 365
                )

                yearly_angle = (
                    2 * np.pi
                    * (local_time.dt.dayofyear - 1)
                    / days_in_year
                )

                chunk["time_of_day_sin"] = np.sin(daily_angle)
                chunk["time_of_day_cos"] = np.cos(daily_angle)

                chunk["day_of_year_sin"] = np.sin(yearly_angle)
                chunk["day_of_year_cos"] = np.cos(yearly_angle)

                chunk.to_csv(
                    temporary_file,
                    index=False,
                    mode="w" if first_chunk else "a",
                    header=first_chunk,
                    encoding="utf-8",
                )

                first_chunk = False
                total_rows += len(chunk)

        if first_chunk:
            raise ValueError(f"No data rows in {input_file.name}")

        temporary_file.replace(output_file)

    finally:
        if temporary_file.exists():
            temporary_file.unlink()

    print(f"Saved {total_rows:,} rows: {output_file}")

Saved 2,769,082 rows: C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\solar_5min_with_time_features_03 combine_wether_data.csv
Saved 924,275 rows: C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data\solar_15min_with_time_features_03 combine_wether_data.csv
